# Stage B v5.3 — all 10 queries (generalization of v5.2)

**v5.2 single-query result on val_009 (the worst v4 query)**:
  - K-cap F1 = 0.357 (K=14) and 0.417 (K=10), up from v4's 0.000
  - 7 of 10 gold caught, 216 picks total

**v5.3**: apply the same architecture to all 10 val queries.

Per query: 3-sample Pass-1 → union landscape → Pass-2 per candidate with strict A/B/C/D rules → composite confidence.

**Expected wall time**: ~75-90 min on Colab Blackwell.
**Target macro F1 vs v4 baseline of 0.057**: hoping for 0.20-0.35 macro (lift trajectory of val_009 generalized).

## Phase 0 — Setup

In [ ]:
import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run(["pip","install","-q","-U",
        "vllm>=0.9.1","transformers>=4.51.0","pandas==2.2.3",
        "pyarrow==16.1.0","numpy==1.26.4","tqdm"], check=True)


## Phase 1 — Paths and global config

In [ ]:
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN  = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV     = DRIVE_ROOT / "data"     / "val.csv"
GOLD_SETS   = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json"
VAL_ASPECTS = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"

OUT_DIR = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "v53_all_queries"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K               = 2000
LLM_MODEL           = "Qwen/Qwen3-32B-AWQ"
PASS1_SEEDS         = [42, 43, 44]
PASS1_MAX_TOKENS    = 4096
PASS2_MAX_TOKENS    = 1024
MAX_MODEL_LEN       = 8192
CONF_FLOOR          = 0.50

print(f"Output dir: {OUT_DIR}")
print("\nVerifying inputs:")
for p in [STAGE_B_IN, VAL_CSV, GOLD_SETS, VAL_ASPECTS]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")


## Phase 2 — Load queries, aspects, gold, candidates

In [ ]:
import pandas as pd

val_df = pd.read_csv(VAL_CSV)
QIDS = sorted(val_df.query_id.unique().tolist())
qid_to_query = {r.query_id: str(r.query) for r in val_df.itertuples()}
print(f"Queries: {QIDS}")

asp_df = pd.read_parquet(VAL_ASPECTS)
qid_to_aspects = {r.query_id: list(r.aspects) for r in asp_df.itertuples()}

gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))
qid_to_gold = {q: set(gold_sets.get(q, [])) for q in QIDS}
gold_totals = {q: len(qid_to_gold[q]) for q in QIDS}
print(f"\nGold counts: {gold_totals}")

sb_all = pd.read_parquet(STAGE_B_IN)

# Per-query top-K filter + tier assignment
sb_per_query = {}
for q in QIDS:
    sub = (sb_all[sb_all.qid == q].sort_values("stage_a_rank").head(TOP_K).reset_index(drop=True))
    sub["tier"] = "drop"
    _auto = sub.article_match & ((sub.co_citation_count >= 30) | sub.code_in_target)
    sub.loc[_auto, "tier"] = "auto"
    _mid = (~_auto) & (sub.article_match | (sub.co_citation_count >= 5) | (sub.concept_cosine_score >= 0.55))
    sub.loc[_mid, "tier"] = "llm"
    sb_per_query[q] = sub

print(f"\nTier breakdown per query:")
print(f"{'qid':<8} {'gold_total':>10} {'in_topk':>8} {'auto':>5} {'llm':>5} {'drop':>5}")
for q in QIDS:
    s = sb_per_query[q]
    print(f"  {q:<6} {gold_totals[q]:>10} {int(s.is_gold.sum()):>8} "
          f"{int((s.tier=='auto').sum()):>5} {int((s.tier=='llm').sum()):>5} {int((s.tier=='drop').sum()):>5}")


## Phase 3 — Build Pass-1 prompts (one per query × 3 seeds = 30 prompts)

In [ ]:
PASS1_PROMPT = """You are a senior Swiss attorney with deep practical knowledge of every branch of Swiss federal law (ZGB, OR, StGB, StPO, ZPO, BGG, SchKG, BV, EMRK and the BGE jurisprudence). A client has come to you with the legal question below.

CRITICAL CITATION RULES:
- Cite ONLY Swiss legal provisions you are confident actually exist. Never invent article numbers.
- Code-size limits for Swiss federal codes:
    ZGB:   articles 1-977
    OR:    articles 1-1186
    StGB:  articles 1-393
    StPO:  articles 1-457
    ZPO:   articles 1-408     (anything > 408 is NOT a real ZPO article)
    BGG:   articles 1-132     (anything > 132 is NOT a real BGG article)
    SchKG: articles 1-340
    BV:    articles 1-197
    EMRK:  articles 1-59
- If you do not know the exact article number, OMIT it. A sparse, accurate list is far more useful than a long list with fabricated articles.

LEGAL QUESTION FROM THE CLIENT:
{question}

THE QUESTION'S LEGAL ASPECTS (each is one sub-domain you must map):
{aspects_block}

YOUR TASK:
For EACH aspect, list the canonical Swiss legal authorities a competent Swiss lawyer would cite. Limit each category to AT MOST 5 entries — prioritize the most foundational and well-established.

Categories (omit empty ones):
1. FOUNDATIONAL STATUTES (rules that directly establish the doctrine)
2. CONSTITUTIONAL/TREATY PROVISIONS (BV or EMRK, only if fundamental rights are at stake)
3. PROCEDURAL RULES (e.g., Art. 100 BGG for any Federal Tribunal appeal)
4. LEADING BGE PRECEDENTS (specific BGE references — omit if uncertain)
5. ADJACENT PROVISIONS (articles routinely co-cited in the same statutory neighborhood)

OUTPUT STRICT JSON (no prose, begin with `{{`):
{{
  "a1": {{
    "aspect_label": "<copy from input>",
    "foundational_statutes": [{{"cit": "Art. ... ", "role": "..."}}],
    "constitutional_provisions": [],
    "procedural_rules": [],
    "leading_precedents": [],
    "adjacent_provisions": []
  }},
  "a2": {{...}}, "a3": {{...}}, "a4": {{...}}
}}
"""

def format_aspects_block(aspects):
    parts = []
    for a in aspects:
        aid = a.get("id","")
        lbl = a.get("label","")
        w   = float(a.get("weight",0))
        terms_en = ", ".join(list(a.get("concepts_en",[])))
        terms_de = ", ".join(list(a.get("terms_de",[])))
        terms_fr = ", ".join(list(a.get("terms_fr",[])))
        terms_it = ", ".join(list(a.get("terms_it",[])))
        parts.append(
            f"  Aspect {aid} (weight={w:.2f}): {lbl}\n"
            f"    English concepts: {terms_en}\n"
            f"    German terms:     {terms_de}\n"
            f"    French terms:     {terms_fr}\n"
            f"    Italian terms:    {terms_it}"
        )
    return "\n".join(parts)

pass1_prompts = []
pass1_meta = []        # (qid, seed) parallel to pass1_prompts
for q in QIDS:
    prompt = PASS1_PROMPT.format(
        question=qid_to_query[q],
        aspects_block=format_aspects_block(qid_to_aspects.get(q, [])),
    )
    for seed in PASS1_SEEDS:
        pass1_prompts.append(prompt)
        pass1_meta.append((q, seed))

print(f"Pass-1 prompts: {len(pass1_prompts)}  ({len(QIDS)} queries × {len(PASS1_SEEDS)} seeds)")


## Phase 4 — Run Pass-1 (batched across all queries)

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {LLM_MODEL} ...")
llm = LLM(
    model=LLM_MODEL, dtype="bfloat16",
    gpu_memory_utilization=0.60, max_model_len=MAX_MODEL_LEN,
    enforce_eager=False,
)

# Per-prompt seed control — build one SamplingParams per prompt
sp_list = [SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS1_MAX_TOKENS, seed=seed)
           for _, seed in pass1_meta]

print(f"\nRunning Pass-1 ({len(pass1_prompts)} prompts batched) ...")
t0 = time.time()
# vLLM accepts a list of SamplingParams parallel to prompts
messages = [[{"role": "user", "content": p}] for p in pass1_prompts]
outs1 = llm.chat(messages, sampling_params=sp_list)
pass1_raws = [o.outputs[0].text for o in outs1]
print(f"Pass-1 done in {(time.time()-t0)/60:.1f} min  ({len(pass1_prompts)/max(1,time.time()-t0):.1f} prompts/sec)")


## Phase 5 — Parse + merge landscapes per query

In [ ]:
def parse_landscape(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

def coerce_item(it):
    """Normalize Pass-1 entries: some samples emit plain strings instead of
    {'cit': ..., 'role': ...}; some use alternative key names."""
    if isinstance(it, str):
        return {"cit": it, "role": ""}
    if isinstance(it, dict):
        cit  = it.get("cit") or it.get("citation") or it.get("article") or ""
        role = it.get("role") or it.get("doctrine") or it.get("topic") or it.get("description") or ""
        return {"cit": str(cit), "role": str(role)}
    return None

def norm_cit(c):
    s = re.sub(r"\bAbs\.\s*\d+\w*\b", "", c or "", flags=re.IGNORECASE)
    s = re.sub(r"\blit\.\s*\w+\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"[.,]", "", s)
    return " ".join(s.lower().split())

CATEGORIES = ("foundational_statutes","constitutional_provisions",
              "procedural_rules","leading_precedents","adjacent_provisions")
ART_LIMITS = {"ZGB":977,"OR":1186,"StGB":393,"StPO":457,"ZPO":408,
              "BGG":132,"SchKG":340,"BV":197,"EMRK":59}
ART_RE = re.compile(r"Art\.\s*(\d+)\s*(?:Abs\.\s*\d+\w*\s*)?(?:lit\.\s*\w+\s*)?(\w+)")

def is_valid(cit):
    m = ART_RE.search(cit or "")
    if not m: return True
    num, code = int(m.group(1)), m.group(2)
    return num <= ART_LIMITS.get(code, 99999)

qid_to_landscapes = {q: [] for q in QIDS}
parse_fail = 0
for (qid, seed), raw in zip(pass1_meta, pass1_raws):
    ls = parse_landscape(raw)
    if ls is None:
        parse_fail += 1
        continue
    qid_to_landscapes[qid].append(ls)
print(f"Parsed: {sum(len(v) for v in qid_to_landscapes.values())} ok, {parse_fail} fail")

qid_to_landscape = {}
total_stripped = 0
total_coerced_strings = 0
total_dropped = 0
for q in QIDS:
    samples = qid_to_landscapes[q]
    if not samples:
        qid_to_landscape[q] = {}
        continue
    merged = {}
    for ls in samples:
        if not isinstance(ls, dict): continue
        for aid, asp in ls.items():
            if not isinstance(asp, dict): continue
            if aid not in merged:
                merged[aid] = {"aspect_label": asp.get("aspect_label",""), **{c: {} for c in CATEGORIES}}
            for cat in CATEGORIES:
                items = asp.get(cat, []) or []
                if not isinstance(items, list): continue
                for raw_it in items:
                    if isinstance(raw_it, str):
                        total_coerced_strings += 1
                    it = coerce_item(raw_it)
                    if it is None or not it["cit"]:
                        total_dropped += 1
                        continue
                    key = norm_cit(it["cit"])
                    if key not in merged[aid][cat]:
                        merged[aid][cat][key] = it
    final = {}
    for aid, asp in merged.items():
        final[aid] = {"aspect_label": asp["aspect_label"]}
        for cat in CATEGORIES:
            items = list(asp[cat].values())
            kept = [it for it in items if is_valid(it.get("cit",""))]
            total_stripped += len(items) - len(kept)
            final[aid][cat] = kept
    qid_to_landscape[q] = final

print(f"Coerced {total_coerced_strings} plain-string items to dict form")
print(f"Dropped {total_dropped} unrecognized-shape items")
print(f"Stripped {total_stripped} hallucinated refs across all queries")
print(f"\nLandscape sizes per query:")
print(f"{'qid':<8} {'a1':>4} {'a2':>4} {'a3':>4} {'a4':>4}  total")
for q in QIDS:
    ls = qid_to_landscape[q]
    counts = []
    for aid in ("a1","a2","a3","a4"):
        n = sum(len(ls.get(aid,{}).get(c,[])) for c in CATEGORIES)
        counts.append(n)
    print(f"  {q:<6} {counts[0]:>4} {counts[1]:>4} {counts[2]:>4} {counts[3]:>4}  {sum(counts)}")


## Phase 6 — Build Pass-2 prompts (one per LLM-tier candidate, with that query's landscape)

In [ ]:
def render_landscape(landscape):
    if not landscape: return "(no landscape — Pass-1 failed for this query)"
    chunks = []
    for aid in sorted(landscape.keys()):
        asp = landscape[aid]
        chunks.append(f"== {aid}: {asp.get('aspect_label','')} ==")
        for cat, hdr in [("foundational_statutes","Foundational statutes"),
                         ("constitutional_provisions","Constitutional/treaty"),
                         ("procedural_rules","Procedural rules"),
                         ("leading_precedents","Leading BGE precedents"),
                         ("adjacent_provisions","Adjacent provisions")]:
            items = asp.get(cat, [])
            if not items: continue
            chunks.append(f"  {hdr}:")
            for it in items:
                cit, role = it.get("cit",""), it.get("role","") or it.get("doctrine","")
                chunks.append(f"    - {cit:<25}  {role[:100]}")
    return "\n".join(chunks)

def aspect_terms_block(aspects):
    blocks = []
    for a in aspects:
        aid = a.get("id","")
        terms = list(a.get("terms_de",[])) + list(a.get("terms_fr",[])) + list(a.get("terms_it",[])) + list(a.get("concepts_en",[]))
        blocks.append(f"  {aid}: " + " | ".join(t for t in terms if t))
    return "\n".join(blocks)

PASS2_PROMPT = """You are a senior Swiss lawyer evaluating whether to cite this candidate source. You have already mapped the canonical landscape (below). Apply the decision rules strictly — do not over-include.

LEGAL QUESTION:
{question}

CANONICAL SWISS LEGAL LANDSCAPE (your prior research):
{landscape}

PER-ASPECT KEY TERMINOLOGY (used to validate adjacent_provision claims):
{aspect_terms}

CANDIDATE SOURCE:
- Citation: {citation}
- Type: {family_label}
- {family_extra}
- Paragraph role: {role}
- Substantive text (original language):
---BEGIN---
{text}
---END---

RETRIEVAL EVIDENCE (advisory only):
- Query names this article: {article_match}  | co-citations: {co_citation_count}  | concept-cosine: {concept_cosine_score:.2f}  | code matches area: {code_in_target}

STRICT DECISION RULES (apply in order):

RULE A — Exact landscape match. Does the candidate's CITATION appear (by article number) in foundational/constitutional/procedural/precedential of any aspect?
  - YES -> KEEP, legal_role = matching category, confidence >= 0.85.

RULE B — Listed adjacent. Does the citation appear in adjacent_provisions of any aspect?
  - YES -> KEEP, legal_role = "adjacent_provision", confidence 0.65-0.80.

RULE C — Unlisted but textually on-point. Citation isn't in landscape BUT text directly addresses an aspect's KEY TERMINOLOGY AND states a rule about that exact subject (not just mentions the term).
  - YES -> KEEP, legal_role = "adjacent_provision", confidence 0.50-0.65. matched_aspect must be specific.
  - If you find yourself applying Rule C to most candidates, you are being too permissive — reject instead.

RULE D — Default REJECT. None of A/B/C apply.
  - legal_role = "off_topic", confidence <= 0.30.

OUTPUT STRICT JSON (no prose):
{{"keep": true|false, "confidence": <0.0-1.0>, "matched_aspect": "<a1|a2|a3|a4|none>", "legal_role": "<foundational_statute|constitutional|procedural_rule|leading_precedent|adjacent_provision|off_topic>", "rule_applied": "<A|B|C|D>", "reasoning": "<one sentence>"}}
"""

def family_label(r): return "Swiss court precedent paragraph" if r.family=="court" else "Swiss statutory provision"
def family_extra(r):
    if r.family=="court": return f"Court base: {r.court_base!s}. Chamber: {r.chamber!s}."
    return f"Code: {r.law_code!s}. Law: {r.law_title!s}."

def build_pass2_prompt(r, q, landscape_text, aspect_terms_text):
    return PASS2_PROMPT.format(
        question=qid_to_query[q][:1500],
        landscape=landscape_text,
        aspect_terms=aspect_terms_text,
        citation=r.citation,
        family_label=family_label(r),
        family_extra=family_extra(r),
        role=r.role or "(unknown)",
        article_match=str(bool(r.article_match)).lower(),
        co_citation_count=int(r.co_citation_count),
        concept_cosine_score=float(r.concept_cosine_score),
        code_in_target=str(bool(r.code_in_target)).lower(),
        text=(r.text or "")[:1200].replace('"', "'"),
    )

pass2_prompts = []
pass2_meta = []   # (qid, did) parallel
for q in QIDS:
    landscape_text = render_landscape(qid_to_landscape.get(q, {}))
    aspect_terms_text = aspect_terms_block(qid_to_aspects.get(q, []))
    sb_llm = sb_per_query[q][sb_per_query[q].tier == "llm"].reset_index(drop=True)
    for r in sb_llm.itertuples():
        pass2_prompts.append(build_pass2_prompt(r, q, landscape_text, aspect_terms_text))
        pass2_meta.append((q, r.did))

print(f"Pass-2 prompts: {len(pass2_prompts)}  (across {len(QIDS)} queries)")
print(f"Mean prompt length: {int(sum(len(p) for p in pass2_prompts)/max(1,len(pass2_prompts)))}")


## Phase 7 — Run Pass-2 (batched)

In [ ]:
sp_pass2 = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS2_MAX_TOKENS, seed=42)

print(f"Pass-2 on {len(pass2_prompts):,} candidates ...")
t0 = time.time()
messages = [[{"role": "user", "content": p}] for p in pass2_prompts]
outs2 = llm.chat(messages, sampling_params=sp_pass2)
pass2_raws = [o.outputs[0].text for o in outs2]
print(f"Pass-2 done in {(time.time()-t0)/60:.1f} min ({len(pass2_prompts)/max(1,time.time()-t0):.1f} c/s)")

del llm; gc.collect()
import torch; torch.cuda.empty_cache()


## Phase 8 — Parse Pass-2 + combine AUTO + LLM per query, compute composite confidence

In [ ]:
def parse_pass2(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

def auto_confidence(cc):
    return min(0.85, 0.50 + 0.35 * math.log(max(1, cc) + 1) / 10.0)

# Build (qid, did) -> raw_text lookup
pass2_lookup = {(q, d): r for (q, d), r in zip(pass2_meta, pass2_raws)}

all_rows = []
for q in QIDS:
    sb = sb_per_query[q]
    # AUTO rows
    for r in sb[sb.tier=="auto"].itertuples():
        all_rows.append({
            "qid": q, "did": r.did, "is_gold": bool(r.is_gold), "tier": "auto",
            "stage_a_rank": int(r.stage_a_rank), "citation": r.citation, "family": r.family,
            "keep_llm": True, "confidence": auto_confidence(int(r.co_citation_count)),
            "matched_aspect": "auto", "legal_role": "auto_dossier",
            "rule_applied": "auto", "reasoning": f"strong dossier (cc={int(r.co_citation_count)})",
            "keep_final": True, "parse_ok": True,
        })
    # LLM rows
    for r in sb[sb.tier=="llm"].itertuples():
        raw = pass2_lookup.get((q, r.did), "")
        parsed = parse_pass2(raw) or {}
        keep_llm = bool(parsed.get("keep", False))
        conf = float(parsed.get("confidence", 0.0) or 0.0)
        matched = str(parsed.get("matched_aspect","") or "")[:8]
        legal_role = str(parsed.get("legal_role","") or "")[:60]
        rule = str(parsed.get("rule_applied","") or "")[:4]
        reasoning = str(parsed.get("reasoning","") or "")[:300]
        keep_final = keep_llm and (conf >= CONF_FLOOR)
        all_rows.append({
            "qid": q, "did": r.did, "is_gold": bool(r.is_gold), "tier": "llm",
            "stage_a_rank": int(r.stage_a_rank), "citation": r.citation, "family": r.family,
            "keep_llm": keep_llm, "confidence": conf,
            "matched_aspect": matched, "legal_role": legal_role,
            "rule_applied": rule, "reasoning": reasoning,
            "keep_final": keep_final, "parse_ok": bool(parsed),
        })

out_df = pd.DataFrame(all_rows)
print(f"Total rows: {len(out_df)}")
print(f"Per-tier parse_ok / keep / kept_final:")
for tier in ("auto","llm","drop"):
    sub = out_df[out_df.tier == tier] if tier != "drop" else None
    if sub is None or len(sub) == 0:
        print(f"  {tier}: (none)")
        continue
    print(f"  {tier}: n={len(sub)}, kept={int(sub.keep_final.sum())}, "
          f"mean_conf={sub.confidence.mean():.2f}, parse_ok={sub.parse_ok.mean():.1%}")
print(f"\nRule distribution (LLM tier only):")
print(out_df[out_df.tier=="llm"].rule_applied.value_counts().to_string())


## Phase 9 — Per-query F1 (no cap, K-cap by true gold, K-cap by gold-in-topk)

In [ ]:
def f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)

v4_per_query = {
    "val_001":0.108,"val_002":0.047,"val_003":0.018,"val_004":0.076,"val_005":0.067,
    "val_006":0.109,"val_007":0.066,"val_008":0.013,"val_009":0.000,"val_010":0.065,
}
v4_macro = 0.057

per_query = {}
print(f"{'qid':<8} {'gold':>5} {'in_topk':>8} {'picks':>6} {'correct':>8} "
      f"{'P':>5} {'R':>5} {'F1':>5}  {'F1_K14':>7} {'F1_K10':>7}  {'v4':>5} delta")

for q in QIDS:
    sb = sb_per_query[q]
    gold_total = gold_totals[q]
    gold_in_topk = int(sb.is_gold.sum())
    sub = out_df[out_df.qid == q]
    kept = sub[sub.keep_final]
    picks = set(kept.did)
    correct = picks & qid_to_gold[q]
    p = len(correct) / max(1, len(picks))
    r = len(correct) / max(1, gold_total)
    f1_nc = f1(p, r)
    # K-cap (K=gold_total)
    kept_sorted = kept.sort_values("confidence", ascending=False)
    topK = set(kept_sorted.head(gold_total).did)
    correctK = topK & qid_to_gold[q]
    pK = len(correctK)/max(1, len(topK)); rK = len(correctK)/max(1, gold_total)
    f1_K = f1(pK, rK)
    # K-cap (K=gold_in_topk)
    topK2 = set(kept_sorted.head(gold_in_topk).did)
    correctK2 = topK2 & qid_to_gold[q]
    pK2 = len(correctK2)/max(1, len(topK2)); rK2 = len(correctK2)/max(1, gold_total)
    f1_K2 = f1(pK2, rK2)

    per_query[q] = {"gold": gold_total, "gold_in_topk": gold_in_topk,
                    "picks": len(picks), "correct": len(correct),
                    "P_no_cap": p, "R_no_cap": r, "F1_no_cap": f1_nc,
                    "P_k14": pK, "R_k14": rK, "F1_k14": f1_K,
                    "P_k10": pK2, "R_k10": rK2, "F1_k10": f1_K2,
                    "v4_F1": v4_per_query[q]}

    delta = f1_K - v4_per_query[q]
    sign = "+" if delta > 0 else ""
    print(f"  {q:<6} {gold_total:>5} {gold_in_topk:>8} {len(picks):>6} {len(correct):>8} "
          f"{p:>.3f} {r:>.3f} {f1_nc:>.3f}  {f1_K:>.3f} {f1_K2:>.3f}  "
          f"{v4_per_query[q]:>.3f} {sign}{delta:+.3f}")

macro_p_nc = sum(per_query[q]["P_no_cap"] for q in QIDS)/len(QIDS)
macro_r_nc = sum(per_query[q]["R_no_cap"] for q in QIDS)/len(QIDS)
macro_f_nc = sum(per_query[q]["F1_no_cap"] for q in QIDS)/len(QIDS)
macro_f_K = sum(per_query[q]["F1_k14"] for q in QIDS)/len(QIDS)
macro_f_K2 = sum(per_query[q]["F1_k10"] for q in QIDS)/len(QIDS)

print(f"\nMACRO  P={macro_p_nc:.3f}  R={macro_r_nc:.3f}  F1_no_cap={macro_f_nc:.3f}")
print(f"MACRO  F1_K=gold={macro_f_K:.3f}  F1_K=gold_in_topk={macro_f_K2:.3f}")
print(f"\nvs v4 macro F1=0.057:  no-cap delta={macro_f_nc-v4_macro:+.3f}, K-cap delta={macro_f_K-v4_macro:+.3f}")


## Phase 10 — Save outputs

In [ ]:
out_df.to_parquet(OUT_DIR / "v53_raw.parquet", index=False)
out_df[out_df.keep_final].to_parquet(OUT_DIR / "v53_survivors.parquet", index=False)
with open(OUT_DIR / "v53_landscapes.json", "w", encoding="utf-8") as f:
    json.dump(qid_to_landscape, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "v53_metrics.json", "w") as f:
    json.dump({
        "config": {"TOP_K": TOP_K, "LLM_MODEL": LLM_MODEL,
                   "PASS1_SEEDS": PASS1_SEEDS, "CONF_FLOOR": CONF_FLOOR},
        "macro_F1_no_cap": macro_f_nc,
        "macro_F1_K_eq_gold": macro_f_K,
        "macro_F1_K_eq_gold_in_topk": macro_f_K2,
        "per_query": per_query,
        "v4_macro_baseline": v4_macro,
    }, f, indent=2)
print(f"Saved 4 files to {OUT_DIR}")
print(f"\nv5.3 macro F1: no-cap={macro_f_nc:.3f}, K=gold={macro_f_K:.3f}, K=gold_in_topk={macro_f_K2:.3f}")
print(f"v4 baseline: 0.057")
